Unlike traditional neural networks, PINNs don't require large datasets of input-output pairs. Instead, they are trained using:

A set of time points (our only "input data")
The physics equations themselves (encoded in the loss function)
Initial/boundary conditions

# Our only "training data" is just time points
t = torch.linspace(0, 2, 100).reshape(-1, 1)  # 100 time points from 0 to 2 seconds
t0 = torch.zeros(1, 1)  # Single point for initial conditions

The PINN learns from:

Physics Loss: For each time point t, we enforce

def physics_loss(model, t):
    _, _, a = model.compute_derivatives(t)
    # Horizontal acceleration should be 0
    ax_loss = torch.mean(torch.square(a[:, 0]))
    # Vertical acceleration should be -g
    ay_loss = torch.mean(torch.square(a[:, 1] + model.g))
    return ax_loss + ay_loss

Initial Conditions: At t=0, we enforce:

def initial_conditions_loss(model, t0):
    xy0, v0_pred, _ = model.compute_derivatives(t0)
    # Position should be (0,0)
    pos_loss = torch.mean(torch.square(xy0[:, 0] - model.x0) +
                          torch.square(xy0[:, 1] - model.y0))
    # Velocity should match initial velocity and angle
    vx0_expected = model.v0 * torch.cos(model.theta)
    vy0_expected = model.v0 * torch.sin(model.theta)
    vel_loss = torch.mean(torch.square(v0_pred[:, 0] - vx0_expected) +
                          torch.square(v0_pred[:, 1] - vy0_expected))
    return pos_loss + vel_loss

# Neural Network Approaches: Traditional vs Physics-Informed

| Characteristic | Traditional Neural Network | Physics-Informed Neural Network (PINN) |
|----------------|---------------------------|---------------------------------------|
| Training Data | Requires large dataset of input-output pairs | Only needs collocation points (e.g., time points) |
| Learning Approach | Learns patterns directly from data | Learns to satisfy differential equations |
| Loss Function | Based on prediction error vs actual data | Based on physics equations and boundary conditions |
| Physical Constraints | May violate physical laws | Physics built into loss function |
| Data Requirements | High - needs many examples | Low - needs only initial/boundary conditions |
| Interpretability | Black box - learns correlations | More interpretable - based on physical laws |
| Extrapolation | Often poor beyond training data | Better - follows physical laws |
| Computational Cost | Lower - simple loss computation | Higher - requires derivative computations |
| Primary Use Case | Pattern recognition from data | Solving differential equations |
| Example Application | Predicting trajectory from historical data | Solving equations of motion from physics |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

class ProjectilePINN(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        # Neural network: 1 input (time) -> 2 outputs (x, y positions)
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 2)
        )

        # Physics constants
        self.g = 9.81  # gravity (m/s^2)

        # Initial conditions
        self.v0 = 20.0  # initial velocity (m/s)
        self.theta = torch.tensor(45.0 * np.pi / 180.0)  # launch angle (radians)
        self.x0 = 0.0  # initial x position
        self.y0 = 0.0  # initial y position

    def forward(self, t):
        return self.net(t)

    def compute_derivatives(self, t):
        # Compute first and second derivatives using autograd
        t.requires_grad_(True)
        xy = self.forward(t)

        # First derivatives (velocities)
        dxy_dt = torch.autograd.grad(
            xy, t,
            grad_outputs=torch.ones_like(xy),
            create_graph=True
        )[0]

        # Reshape velocities to match batch dimension
        vx = dxy_dt * torch.ones_like(xy[:, 0:1])
        vy = dxy_dt * torch.ones_like(xy[:, 1:2])
        v = torch.cat([vx, vy], dim=1)

        # Second derivatives (accelerations)
        d2xy_dt2 = torch.autograd.grad(
            v, t,
            grad_outputs=torch.ones_like(v),
            create_graph=True
        )[0]

        # Reshape accelerations to match batch dimension
        ax = d2xy_dt2 * torch.ones_like(xy[:, 0:1])
        ay = d2xy_dt2 * torch.ones_like(xy[:, 1:2])
        a = torch.cat([ax, ay], dim=1)

        return xy, v, a

def physics_loss(model, t):
    # Get positions, velocities, and accelerations
    _, _, a = model.compute_derivatives(t)

    # Physics equations for projectile motion:
    # d²x/dt² = 0 (no horizontal acceleration)
    # d²y/dt² = -g (constant gravitational acceleration)
    ax_loss = torch.mean(torch.square(a[:, 0]))  # should be 0
    ay_loss = torch.mean(torch.square(a[:, 1] + model.g))  # should be -g

    return ax_loss + ay_loss

def initial_conditions_loss(model, t0):
    xy0, v0_pred, _ = model.compute_derivatives(t0)

    # Initial position loss
    pos_loss = torch.mean(torch.square(xy0[:, 0] - model.x0) +
                          torch.square(xy0[:, 1] - model.y0))

    # Initial velocity loss
    vx0_expected = model.v0 * torch.cos(model.theta)
    vy0_expected = model.v0 * torch.sin(model.theta)
    vel_loss = torch.mean(torch.square(v0_pred[:, 0] - vx0_expected) +
                          torch.square(v0_pred[:, 1] - vy0_expected))

    return pos_loss + vel_loss

# Training setup
model = ProjectilePINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Generate training points
t = torch.linspace(0, 2, 100).reshape(-1, 1)  # 2 seconds of motion
t0 = torch.zeros(1, 1)  # Initial time point

# Training loop
epochs = 5000
for epoch in range(epochs):
    optimizer.zero_grad()

    # Compute losses
    phys_loss = physics_loss(model, t)
    ic_loss = initial_conditions_loss(model, t0)
    total_loss = phys_loss + 100 * ic_loss  # Weight IC loss higher

    # Backpropagation
    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f'Epoch {epoch+1}, Loss: {total_loss.item():.6f}')

# Plotting
with torch.no_grad():
    t_plot = torch.linspace(0, 2, 200).reshape(-1, 1)
    xy_pred = model(t_plot)

    # Analytical solution for comparison
    t_np = t_plot.numpy()
    x_analytical = model.v0 * np.cos(model.theta.item()) * t_np
    y_analytical = (model.v0 * np.sin(model.theta.item()) * t_np -
                    0.5 * model.g * t_np**2)

    plt.figure(figsize=(15, 5))

    # Plot X position over time
    plt.subplot(1, 3, 1)
    plt.plot(t_plot, xy_pred[:, 0].detach(), 'b-', label='PINN')
    plt.plot(t_plot, x_analytical, 'r--', label='Analytical')
    plt.xlabel('Time (s)')
    plt.ylabel('X Position (m)')
    plt.legend()
    plt.grid(True)

    # Plot Y position over time
    plt.subplot(1, 3, 2)
    plt.plot(t_plot, xy_pred[:, 1].detach(), 'b-', label='PINN')
    plt.plot(t_plot, y_analytical, 'r--', label='Analytical')
    plt.xlabel('Time (s)')
    plt.ylabel('Y Position (m)')
    plt.legend()
    plt.grid(True)

    # Plot trajectory
    plt.subplot(1, 3, 3)
    plt.plot(xy_pred[:, 0].detach(), xy_pred[:, 1].detach(), 'b-', label='PINN')
    plt.plot(x_analytical, y_analytical, 'r--', label='Analytical')
    plt.xlabel('X Position (m)')
    plt.ylabel('Y Position (m)')
    plt.title('Projectile Trajectory')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

The x-position over time (left plot) is showing linear motion but at a much slower rate than expected
The y-position over time (middle plot) has the right parabolic shape but doesn't match the analytical solution
The trajectory (right plot) shows the projectile going higher than it should

Let's improve this by:

Adjusting the weight between physics and initial condition losses
Increasing the network capacity
Adding more training points in critical regions
Key improvements made:

Increased network capacity (more layers and neurons)
Added more training points with focus on early trajectory
Increased weight of initial conditions loss (1000 instead of 100)
Increased training epochs to 10000
Added stronger physics constraints multiplier (10.0)
Improved derivative computation with allow_unused=True

This should produce a much better match to the analytical solution. The key insight is that since we're not using real training data, we need to carefully balance:

The physics constraints (accelerations)
The initial conditions (position and velocity at t=0)
The distribution of time points where we enforce these constraints

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

class ProjectilePINN(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 2)
        )

        self.g = 9.81
        self.v0 = 20.0
        self.theta = torch.tensor(45.0 * np.pi / 180.0)
        self.x0 = 0.0
        self.y0 = 0.0

    def forward(self, t):
        return self.net(t)

    def compute_derivatives(self, t):
        t.requires_grad_(True)
        xy = self.forward(t)

        # Compute velocities (first derivatives)
        velocities = []
        for i in range(2):  # For x and y components
            v = torch.autograd.grad(
                xy[:, i].sum(),
                t,
                create_graph=True,
                retain_graph=True
            )[0]
            velocities.append(v)
        v = torch.cat(velocities, dim=1)

        # Compute accelerations (second derivatives)
        accelerations = []
        for i in range(2):  # For x and y components
            a = torch.autograd.grad(
                v[:, i].sum(),
                t,
                create_graph=True,
                retain_graph=True
            )[0]
            accelerations.append(a)
        a = torch.cat(accelerations, dim=1)

        return xy, v, a

def physics_loss(model, t):
    _, _, a = model.compute_derivatives(t)

    # Physics equations: d²x/dt² = 0, d²y/dt² = -g
    ax_loss = 10.0 * torch.mean(torch.square(a[:, 0]))
    ay_loss = 10.0 * torch.mean(torch.square(a[:, 1] + model.g))

    return ax_loss + ay_loss

def initial_conditions_loss(model, t0):
    xy0, v0_pred, _ = model.compute_derivatives(t0)

    # Position at t=0 should be (0,0)
    pos_loss = torch.mean(torch.square(xy0[:, 0] - model.x0) +
                          torch.square(xy0[:, 1] - model.y0))

    # Initial velocities should match v0 and angle
    vx0_expected = model.v0 * torch.cos(model.theta)
    vy0_expected = model.v0 * torch.sin(model.theta)
    vel_loss = torch.mean(torch.square(v0_pred[:, 0] - vx0_expected) +
                          torch.square(v0_pred[:, 1] - vy0_expected))

    return pos_loss + vel_loss

# Set up model and optimizer
model = ProjectilePINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Generate training points with focus on early trajectory
t1 = torch.linspace(0, 0.5, 50).reshape(-1, 1)
t2 = torch.linspace(0.5, 2, 50).reshape(-1, 1)
t = torch.cat([t1, t2])
t0 = torch.zeros(1, 1)

# Training loop
epochs = 10000
for epoch in range(epochs):
    optimizer.zero_grad()

    phys_loss = physics_loss(model, t)
    ic_loss = initial_conditions_loss(model, t0)
    total_loss = phys_loss + 1000 * ic_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f'Epoch {epoch+1}, Loss: {total_loss.item():.6f}')

# Plotting
with torch.no_grad():
    t_plot = torch.linspace(0, 2, 200).reshape(-1, 1)
    xy_pred = model(t_plot)

    t_np = t_plot.numpy()
    x_analytical = model.v0 * np.cos(model.theta.item()) * t_np
    y_analytical = (model.v0 * np.sin(model.theta.item()) * t_np -
                    0.5 * model.g * t_np**2)

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.plot(t_plot, xy_pred[:, 0].detach(), 'b-', label='PINN')
    plt.plot(t_plot, x_analytical, 'r--', label='Analytical')
    plt.xlabel('Time (s)')
    plt.ylabel('X Position (m)')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 2)
    plt.plot(t_plot, xy_pred[:, 1].detach(), 'b-', label='PINN')
    plt.plot(t_plot, y_analytical, 'r--', label='Analytical')
    plt.xlabel('Time (s)')
    plt.ylabel('Y Position (m)')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 3, 3)
    plt.plot(xy_pred[:, 0].detach(), xy_pred[:, 1].detach(), 'b-', label='PINN')
    plt.plot(x_analytical, y_analytical, 'r--', label='Analytical')
    plt.xlabel('X Position (m)')
    plt.ylabel('Y Position (m)')
    plt.title('Projectile Trajectory')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

Next example is a nonlinear pendulum with air resistance.
This is a great example because:

It has nonlinear dynamics (sin term in pendulum motion)
Includes velocity-dependent damping (quadratic air resistance)
No simple analytical solution exists
Multiple coupled differential equations

Nonlinear Terms:

Full nonlinear pendulum equation (using sin(θ) instead of small angle approximation)
Quadratic air resistance (proportional to velocity squared)


Coupled Equations:

Angular position (θ) and angular velocity (ω) are coupled
Second-order ODE split into two first-order ODEs


Complex Physics:

Gravitational torque: g/L * sin(θ)
Air resistance: Cd * ρ * A * L * |ω| * ω / (2m)
Full rotational dynamics


Visualization:

Angular position over time
Angular velocity over time
Phase space trajectory (θ vs ω)
Pendulum motion path



The system is governed by the differential equation:
d²θ/dt² + (CdρA*L/2m)|dθ/dt|*dθ/dt + (g/L)sin(θ) = 0

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

class NonlinearPendulumPINN(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 2)  # outputs [theta, omega]
        )

        # Physical parameters
        self.g = 9.81      # gravity (m/s^2)
        self.L = 1.0       # pendulum length (m)
        self.m = 1.0       # mass (kg)
        self.Cd = 0.1      # drag coefficient
        self.rho = 1.225   # air density (kg/m^3)
        self.A = 0.01      # cross-sectional area (m^2)

        # Initial conditions
        self.theta0 = torch.tensor(np.pi/2)  # Initial angle (90 degrees)
        self.omega0 = torch.tensor(0.0)      # Initial angular velocity

    def forward(self, t):
        return self.net(t)

    def compute_derivatives(self, t):
        t.requires_grad_(True)
        theta_omega = self.forward(t)
        theta, omega = theta_omega[:, 0:1], theta_omega[:, 1:2]

        # First derivatives
        dtheta_dt = torch.autograd.grad(
            theta.sum(), t,
            create_graph=True,
            retain_graph=True
        )[0]

        domega_dt = torch.autograd.grad(
            omega.sum(), t,
            create_graph=True,
            retain_graph=True
        )[0]

        return theta_omega, dtheta_dt, domega_dt

def physics_loss(model, t):
    theta_omega, dtheta_dt, domega_dt = model.compute_derivatives(t)
    theta, omega = theta_omega[:, 0:1], theta_omega[:, 1:2]

    # Nonlinear pendulum equation with quadratic air resistance
    # d²θ/dt² + (Cd*ρ*A*L/2m)*|dθ/dt|*dθ/dt + (g/L)*sin(θ) = 0

    # Air resistance term
    air_resistance = (model.Cd * model.rho * model.A * model.L / (2 * model.m)) * \
                     torch.abs(omega) * omega

    # Gravitational term
    gravity_term = (model.g / model.L) * torch.sin(theta)

    # Physics residual: dω/dt + air_resistance + g/L*sin(θ) = 0
    physics_residual = domega_dt + air_resistance + gravity_term

    # Kinematic constraint: dθ/dt = ω
    kinematic_residual = dtheta_dt - omega

    return torch.mean(torch.square(physics_residual)) + \
        torch.mean(torch.square(kinematic_residual))

def initial_conditions_loss(model, t0):
    theta_omega, _, _ = model.compute_derivatives(t0)
    theta0_pred, omega0_pred = theta_omega[:, 0], theta_omega[:, 1]

    ic_loss = torch.mean(torch.square(theta0_pred - model.theta0)) + \
              torch.mean(torch.square(omega0_pred - model.omega0))

    return ic_loss

# Setup model and optimizer
model = NonlinearPendulumPINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Generate training points with focus on early dynamics
t1 = torch.linspace(0, 1, 100).reshape(-1, 1)   # More points early
t2 = torch.linspace(1, 5, 100).reshape(-1, 1)   # Fewer points later
t = torch.cat([t1, t2])
t0 = torch.zeros(1, 1)

# Training loop
epochs = 20000
for epoch in range(epochs):
    optimizer.zero_grad()

    phys_loss = physics_loss(model, t)
    ic_loss = initial_conditions_loss(model, t0)
    total_loss = phys_loss + 1000 * ic_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}, Loss: {total_loss.item():.6f}')

# Generate predictions
with torch.no_grad():
    t_plot = torch.linspace(0, 5, 500).reshape(-1, 1)
    predictions = model(t_plot)
    theta_pred = predictions[:, 0].numpy()
    omega_pred = predictions[:, 1].numpy()
    t_np = t_plot.numpy().flatten()

    # Convert to Cartesian coordinates for visualization
    x = model.L * np.sin(theta_pred)
    y = -model.L * np.cos(theta_pred)

    # Create plots
    fig = plt.figure(figsize=(15, 5))

    # Angular position over time
    plt.subplot(1, 3, 1)
    plt.plot(t_np, theta_pred, 'b-', label='θ(t)')
    plt.xlabel('Time (s)')
    plt.ylabel('Angle (rad)')
    plt.title('Angular Position')
    plt.grid(True)
    plt.legend()

    # Angular velocity over time
    plt.subplot(1, 3, 2)
    plt.plot(t_np, omega_pred, 'r-', label='ω(t)')
    plt.xlabel('Time (s)')
    plt.ylabel('Angular Velocity (rad/s)')
    plt.title('Angular Velocity')
    plt.grid(True)
    plt.legend()

    # Phase space trajectory
    plt.subplot(1, 3, 3)
    plt.plot(theta_pred, omega_pred, 'g-', label='Phase Space')
    plt.xlabel('θ (rad)')
    plt.ylabel('ω (rad/s)')
    plt.title('Phase Space Trajectory')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Animate pendulum motion
    plt.figure(figsize=(6, 6))
    plt.plot(x, y, 'b-', alpha=0.3, label='Path')
    plt.plot([0, x[0]], [0, y[0]], 'k-', label='Pendulum')
    plt.plot(0, 0, 'ko', label='Pivot')
    plt.xlabel('x (m)')
    plt.ylabel('y (m)')
    plt.title('Pendulum Motion Path')
    plt.grid(True)
    plt.axis('equal')
    plt.legend()
    plt.show()

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

class NonlinearPendulumPINN(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 2)  # outputs [theta, omega]
        )

        # Physical parameters
        self.g = 9.81      # gravity (m/s^2)
        self.L = 1.0       # pendulum length (m)
        self.m = 1.0       # mass (kg)
        self.Cd = 0.1      # drag coefficient
        self.rho = 1.225   # air density (kg/m^3)
        self.A = 0.01      # cross-sectional area (m^2)

        # Initial conditions
        self.theta0 = torch.tensor(np.pi/2)  # Initial angle (90 degrees)
        self.omega0 = torch.tensor(0.0)      # Initial angular velocity

    def forward(self, t):
        return self.net(t)

    def compute_derivatives(self, t):
        t.requires_grad_(True)
        theta_omega = self.forward(t)
        theta, omega = theta_omega[:, 0:1], theta_omega[:, 1:2]

        dtheta_dt = torch.autograd.grad(
            theta.sum(), t,
            create_graph=True,
            retain_graph=True
        )[0]

        domega_dt = torch.autograd.grad(
            omega.sum(), t,
            create_graph=True,
            retain_graph=True
        )[0]

        return theta_omega, dtheta_dt, domega_dt

    def compute_energy(self, theta, omega):
        # Kinetic energy: (1/2)mL²ω²
        T = 0.5 * self.m * self.L**2 * omega**2
        # Potential energy: mgL(1 - cos(θ))
        V = self.m * self.g * self.L * (1 - torch.cos(theta))
        return T + V

def physics_loss(model, t):
    theta_omega, dtheta_dt, domega_dt = model.compute_derivatives(t)
    theta, omega = theta_omega[:, 0:1], theta_omega[:, 1:2]

    # Air resistance term
    air_resistance = (model.Cd * model.rho * model.A * model.L / (2 * model.m)) * \
                     torch.abs(omega) * omega

    # Gravitational term
    gravity_term = (model.g / model.L) * torch.sin(theta)

    physics_residual = domega_dt + air_resistance + gravity_term
    kinematic_residual = dtheta_dt - omega

    return torch.mean(torch.square(physics_residual)) + \
        torch.mean(torch.square(kinematic_residual))

def initial_conditions_loss(model, t0):
    theta_omega, _, _ = model.compute_derivatives(t0)
    theta0_pred, omega0_pred = theta_omega[:, 0], theta_omega[:, 1]

    return torch.mean(torch.square(theta0_pred - model.theta0)) + \
        torch.mean(torch.square(omega0_pred - model.omega0))

# Numerical solution for validation
def pendulum_derivatives(t, state, L, g, m, Cd, rho, A):
    theta, omega = state

    # Air resistance term
    air_resistance = (Cd * rho * A * L / (2 * m)) * abs(omega) * omega

    # System of first-order ODEs
    dtheta_dt = omega
    domega_dt = -(g/L) * np.sin(theta) - air_resistance

    return [dtheta_dt, domega_dt]

# Setup and train PINN
model = NonlinearPendulumPINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training points
t1 = torch.linspace(0, 1, 100).reshape(-1, 1)
t2 = torch.linspace(1, 5, 100).reshape(-1, 1)
t = torch.cat([t1, t2])
t0 = torch.zeros(1, 1)

# Training loop
epochs = 20000
for epoch in range(epochs):
    optimizer.zero_grad()

    phys_loss = physics_loss(model, t)
    ic_loss = initial_conditions_loss(model, t0)
    total_loss = phys_loss + 1000 * ic_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}, Loss: {total_loss.item():.6f}')

# Generate predictions and numerical solution
t_eval = np.linspace(0, 5, 500)
numerical_solution = solve_ivp(
    pendulum_derivatives,
    [0, 5],
    [model.theta0.item(), model.omega0.item()],
    t_eval=t_eval,
    args=(model.L, model.g, model.m, model.Cd, model.rho, model.A),
    method='RK45',
    rtol=1e-8
)

with torch.no_grad():
    t_plot = torch.linspace(0, 5, 500).reshape(-1, 1)
    predictions = model(t_plot)
    theta_pred = predictions[:, 0].numpy()
    omega_pred = predictions[:, 1].numpy()

    # Compute energy
    energy = model.compute_energy(
        torch.tensor(theta_pred),
        torch.tensor(omega_pred)
    ).numpy()

    # Create comparison plots
    fig = plt.figure(figsize=(20, 5))

    # Angular position comparison
    plt.subplot(1, 4, 1)
    plt.plot(t_eval, theta_pred, 'b-', label='PINN')
    plt.plot(t_eval, numerical_solution.y[0], 'r--', label='Numerical')
    plt.xlabel('Time (s)')
    plt.ylabel('Angle (rad)')
    plt.title('Angular Position')
    plt.grid(True)
    plt.legend()

    # Angular velocity comparison
    plt.subplot(1, 4, 2)
    plt.plot(t_eval, omega_pred, 'b-', label='PINN')
    plt.plot(t_eval, numerical_solution.y[1], 'r--', label='Numerical')
    plt.xlabel('Time (s)')
    plt.ylabel('Angular Velocity (rad/s)')
    plt.title('Angular Velocity')
    plt.grid(True)
    plt.legend()

    # Phase space comparison
    plt.subplot(1, 4, 3)
    plt.plot(theta_pred, omega_pred, 'b-', label='PINN')
    plt.plot(numerical_solution.y[0], numerical_solution.y[1], 'r--', label='Numerical')
    plt.xlabel('θ (rad)')
    plt.ylabel('ω (rad/s)')
    plt.title('Phase Space Trajectory')
    plt.grid(True)
    plt.legend()

    # Total energy over time
    plt.subplot(1, 4, 4)
    plt.plot(t_eval, energy, 'g-', label='Total Energy')
    plt.xlabel('Time (s)')
    plt.ylabel('Energy (J)')
    plt.title('Total Energy vs Time')
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

# Calculate average error between PINN and numerical solution
theta_error = np.mean(np.abs(theta_pred - numerical_solution.y[0]))
omega_error = np.mean(np.abs(omega_pred - numerical_solution.y[1]))
print(f"Average absolute error in θ: {theta_error:.6f} rad")
print(f"Average absolute error in ω: {omega_error:.6f} rad/s")

this really illustrates some of the key criticisms of PINNs! Looking at these results:

Accuracy Issues:

The PINN solution (blue) significantly deviates from the numerical solution (red dashed)
Average absolute error in θ is ~0.78 radians (~45 degrees!), which is huge
Angular velocity error of ~1.58 rad/s is also very large


Training Problems:

The loss isn't consistently decreasing (see the fluctuations in the epoch losses)
Even after 20,000 epochs, we haven't achieved good convergence
The phase space trajectory shows the PINN isn't capturing the proper dynamics


Physical Inconsistency:

The PINN solution seems to artificially dampen too quickly
It's not preserving the proper oscillatory behavior
The phase space trajectory (third plot) shows a much smaller and more distorted orbit than it should



This is a perfect example of why many experts are skeptical of PINNs for forward problems:

A traditional numerical solver (the red dashed line) gives much better results with far less computational effort
The PINN is struggling with basic physical behaviors that numerical methods handle easily
Despite the fancy architecture and long training time, we're getting worse results than simple ODE solvers

Limitations/Criticisms:

Training instability - PINNs can be very difficult to train, especially with stiff equations or multiple competing loss terms
Computational cost - calculating derivatives through automatic differentiation is expensive compared to traditional numerical methods
Accuracy concerns - established numerical methods (like finite elements or spectral methods) often achieve better accuracy with less computational cost
Limited advantage - in many cases where analytical solutions exist or traditional numerical methods work well, PINNs offer no clear benefit

Current Valid Use Cases:

Inverse problems - identifying unknown parameters in PDEs from sparse data
High-dimensional PDEs where traditional methods struggle with curse of dimensionality
Problems where we have sparse, noisy data and want to incorporate physical constraints
Situations where we need differentiable surrogates of physical systems

The consensus among many experts seems to be:

PINNs are an interesting research direction but not yet mature for most production applications
Traditional numerical methods are still preferred for most forward problems
PINNs might be most valuable when combined with data (as regularizers or constraints) rather than as pure PDE solvers



First, the traditional numerical solution of our pendulum: 
Traditional ODE Solver:

Takes just ~20 lines of core code
Solves accurately in milliseconds
No training required
Highly reliable results

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

def pendulum_system(t, state, L=1.0, g=9.81, m=1.0, Cd=0.1, rho=1.225, A=0.01):
    theta, omega = state
    
    # Air resistance term
    air_resistance = (Cd * rho * A * L / (2 * m)) * abs(omega) * omega
    
    # System of first-order ODEs
    dtheta_dt = omega
    domega_dt = -(g/L) * np.sin(theta) - air_resistance
    
    return [dtheta_dt, domega_dt]

# Initial conditions
theta0 = np.pi/2  # 90 degrees
omega0 = 0.0      # No initial angular velocity
t_span = [0, 5]   # 5 seconds simulation
t_eval = np.linspace(0, 5, 500)  # Points for evaluation

# Solve using RK45 method
solution = solve_ivp(
    pendulum_system, 
    t_span, 
    [theta0, omega0],
    t_eval=t_eval,
    method='RK45',
    rtol=1e-8
)

# Plot results
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(solution.t, solution.y[0], 'b-', label='θ(t)')
plt.xlabel('Time (s)')
plt.ylabel('Angle (rad)')
plt.title('Pendulum Motion')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(solution.y[0], solution.y[1], 'r-', label='Phase Space')
plt.xlabel('θ (rad)')
plt.ylabel('ω (rad/s)')
plt.title('Phase Space Trajectory')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

print(f"Solution took {solution.nfev} function evaluations")
print(f"Integration completed successfully: {solution.success}")

Now, here's a case where PINNs can be valuable - an inverse problem where we have sparse, noisy measurements and want to identify system parameters:

PINN for Inverse Problem:

Can identify system parameters from sparse, noisy data
Incorporates physics knowledge as regularization
Handles uncertainty in measurements
Provides continuous solution while respecting physical laws
Can identify multiple parameters simultaneously



This illustrates why PINNs are more promising for:

Inverse problems
Cases with sparse/noisy data
Problems where we want to learn parameters
Situations where we need differentiable solutions

Rather than for:

Forward problems with known parameters
Cases where traditional numerical methods work well
Situations requiring high accuracy

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

class InversePendulumPINN(nn.Module):
    def __init__(self, hidden_dim=128):
        super().__init__()
        # Neural network for theta prediction
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 2)  # [theta, omega]
        )
        
        # Learnable parameters
        self.log_L = nn.Parameter(torch.tensor(0.0))  # log of pendulum length
        self.log_Cd = nn.Parameter(torch.tensor(-2.3))  # log of drag coefficient
        
        # Known parameters
        self.g = 9.81
        self.m = 1.0
        self.rho = 1.225
        self.A = 0.01
        
    @property
    def L(self):
        return torch.exp(self.log_L)
    
    @property
    def Cd(self):
        return torch.exp(self.log_Cd)
    
    def forward(self, t):
        return self.net(t)
    
    def compute_derivatives(self, t):
        t.requires_grad_(True)
        theta_omega = self.forward(t)
        theta, omega = theta_omega[:, 0:1], theta_omega[:, 1:2]
        
        dtheta_dt = torch.autograd.grad(
            theta.sum(), t,
            create_graph=True,
            retain_graph=True
        )[0]
        
        domega_dt = torch.autograd.grad(
            omega.sum(), t,
            create_graph=True,
            retain_graph=True
        )[0]
        
        return theta_omega, dtheta_dt, domega_dt

# Generate synthetic sparse, noisy data
true_L = 1.0
true_Cd = 0.1

def generate_noisy_data(n_points=10, noise_level=0.05):
    # Generate clean data using numerical solution
    t_eval = np.linspace(0, 5, 500)
    clean_solution = solve_ivp(
        lambda t, y: pendulum_system(t, y, L=true_L, Cd=true_Cd),
        [0, 5],
        [np.pi/2, 0.0],
        t_eval=t_eval
    )
    
    # Randomly sample n_points and add noise
    idx = np.sort(np.random.choice(len(t_eval), n_points, replace=False))
    t_data = t_eval[idx]
    theta_data = clean_solution.y[0][idx] + noise_level * np.random.randn(n_points)
    
    return t_data, theta_data

# Generate sparse noisy data
t_data, theta_data = generate_noisy_data()
t_tensor = torch.tensor(t_data.reshape(-1, 1), dtype=torch.float32)
theta_tensor = torch.tensor(theta_data.reshape(-1, 1), dtype=torch.float32)

# Training
model = InversePendulumPINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def total_loss(model, t, theta_obs):
    # Physics loss
    theta_omega, dtheta_dt, domega_dt = model.compute_derivatives(t)
    theta, omega = theta_omega[:, 0:1], theta_omega[:, 1:2]
    
    air_resistance = (model.Cd * model.rho * model.A * model.L / (2 * model.m)) * \
                    torch.abs(omega) * omega
    gravity_term = (model.g / model.L) * torch.sin(theta)
    
    physics_residual = domega_dt + air_resistance + gravity_term
    kinematic_residual = dtheta_dt - omega
    
    physics_loss = torch.mean(torch.square(physics_residual)) + \
                  torch.mean(torch.square(kinematic_residual))
    
    # Data loss
    data_loss = torch.mean(torch.square(theta - theta_obs))
    
    return physics_loss + 10.0 * data_loss

# Training loop
epochs = 10000
for epoch in range(epochs):
    optimizer.zero_grad()
    loss = total_loss(model, t_tensor, theta_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}')
        print(f'Loss: {loss.item():.6f}')
        print(f'Predicted L: {model.L.item():.3f} (true: {true_L})')
        print(f'Predicted Cd: {model.Cd.item():.3f} (true: {true_Cd})\n')

# Final results and visualization
with torch.no_grad():
    t_plot = torch.linspace(0, 5, 500).reshape(-1, 1)
    predictions = model(t_plot)
    theta_pred = predictions[:, 0].numpy()
    
    # Generate true solution for comparison
    true_solution = solve_ivp(
        lambda t, y: pendulum_system(t, y, L=true_L, Cd=true_Cd),
        [0, 5],
        [np.pi/2, 0.0],
        t_eval=t_plot.numpy().flatten()
    )
    
    plt.figure(figsize=(10, 6))
    plt.plot(t_plot, theta_pred, 'b-', label='PINN Prediction')
    plt.plot(t_plot, true_solution.y[0], 'r--', label='True Solution')
    plt.plot(t_data, theta_data, 'ko', label='Noisy Data Points')
    plt.xlabel('Time (s)')
    plt.ylabel('Angle (rad)')
    plt.title('Parameter Identification from Sparse, Noisy Data')
    plt.legend()
    plt.grid(True)
    plt.show()

print("\nFinal Results:")
print(f"True Length: {true_L:.3f}, Predicted: {model.L.item():.3f}")
print(f"True Cd: {true_Cd:.3f}, Predicted: {model.Cd.item():.3f}")

Let me explain the different types of data in PINNs:

Pure PINN (what we did first):


Only uses physical equations (PDEs/ODEs)
Initial/boundary conditions
NO measurement data
Learning is purely from physics constraints


Physics-Informed Deep Learning (what I showed in example 2):


Combines physics equations WITH some real measurements
In my example, we had:

In [ ]:
# Generate synthetic sparse, noisy data
def generate_noisy_data(n_points=10, noise_level=0.05):
    # Generate clean data using numerical solution
    t_eval = np.linspace(0, 5, 500)
    clean_solution = solve_ivp(...)

    # Randomly sample n_points and add noise
    idx = np.sort(np.random.choice(len(t_eval), n_points, replace=False))
    t_data = t_eval[idx]
    theta_data = clean_solution.y[0][idx] + noise_level * np.random.randn(n_points)

We used 10 noisy measurements of the pendulum angle
The loss function had both physics and data terms:

In [ ]:
def total_loss(model, t, theta_obs):
    # Physics loss from equations
    physics_loss = torch.mean(torch.square(physics_residual)) + \
                   torch.mean(torch.square(kinematic_residual))

    # Data loss from measurements
    data_loss = torch.mean(torch.square(theta - theta_obs))

    return physics_loss + 10.0 * data_loss

This leads to three main approaches:

Pure PINN: Only physics equations (like our first example)
Pure Data: Traditional neural net with only measurement data
Hybrid: Physics equations + some data (like our second example)

Would you like me to:

Show a comparison of these three approaches?
Demonstrate how the accuracy changes as we vary the amount of measurement data?
Show how different types of measurement data (position vs. velocity vs. both) affect the results?

The key point is that while pure PINNs don't need data, they often work better when combined with some real measurements - this is actually one of their most promising use cases!

This example shows a PINN identifying:

Location of a hidden heat source
Strength of the heat source
Full temperature field

From only:

Sparse temperature measurements (20 sensors)
Known heat equation physics
Boundary conditions

Key advantages of PINNs here:

Can handle sparse, noisy measurements
Incorporates physical constraints
Provides full field reconstruction
Can identify multiple parameters simultaneously
Works with complex geometries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

class HeatSourcePINN(nn.Module):
    def __init__(self, hidden_dim=50):
        super().__init__()
        # Neural network for temperature prediction
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),  # 2D space (x, y)
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)   # Temperature output
        )

        # Learnable parameters for heat source
        self.source_x = nn.Parameter(torch.tensor(0.5))  # x location
        self.source_y = nn.Parameter(torch.tensor(0.5))  # y location
        self.source_strength = nn.Parameter(torch.tensor(1.0))  # strength

        # Known parameters
        self.k = 1.0  # thermal conductivity

    def forward(self, x):
        return self.net(x)

    def source_term(self, x, y):
        # Gaussian heat source
        return self.source_strength * torch.exp(
            -50 * ((x - self.source_x)**2 + (y - self.source_y)**2)
        )

    def compute_derivatives(self, xy):
        x, y = xy[:, 0:1], xy[:, 1:2]
        x.requires_grad_(True)
        y.requires_grad_(True)

        T = self.forward(torch.cat([x, y], dim=1))

        # Compute gradients
        dT_dx = torch.autograd.grad(
            T.sum(), x,
            create_graph=True,
            retain_graph=True
        )[0]

        dT_dy = torch.autograd.grad(
            T.sum(), y,
            create_graph=True,
            retain_graph=True
        )[0]

        # Compute second derivatives
        d2T_dx2 = torch.autograd.grad(
            dT_dx.sum(), x,
            create_graph=True,
            retain_graph=True
        )[0]

        d2T_dy2 = torch.autograd.grad(
            dT_dy.sum(), y,
            create_graph=True,
            retain_graph=True
        )[0]

        return T, d2T_dx2 + d2T_dy2  # Temperature and Laplacian

# Generate synthetic data
def generate_data(true_x=0.7, true_y=0.3, true_strength=2.0, n_sensors=20, noise_level=0.05):
    # Random sensor locations
    x = np.random.uniform(0, 1, n_sensors)
    y = np.random.uniform(0, 1, n_sensors)

    # True temperature (analytical solution for simple case)
    r2 = (x - true_x)**2 + (y - true_y)**2
    T = true_strength * np.exp(-50 * r2)

    # Add noise
    T += noise_level * np.random.randn(n_sensors)

    return x, y, T

# Generate synthetic sensor data
x_data, y_data, T_data = generate_data()
xy_data = torch.tensor(np.stack([x_data, y_data], axis=1), dtype=torch.float32)
T_data = torch.tensor(T_data.reshape(-1, 1), dtype=torch.float32)

# Create mesh for visualization
x_mesh = np.linspace(0, 1, 50)
y_mesh = np.linspace(0, 1, 50)
X, Y = np.meshgrid(x_mesh, y_mesh)
xy_mesh = torch.tensor(np.stack([X.flatten(), Y.flatten()], axis=1), dtype=torch.float32)

# Training
model = HeatSourcePINN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def total_loss(model, xy_mesh, xy_data, T_data):
    # Physics loss (heat equation)
    T_mesh, laplacian = model.compute_derivatives(xy_mesh)
    source = model.source_term(xy_mesh[:, 0:1], xy_mesh[:, 1:2])
    physics_loss = torch.mean(torch.square(model.k * laplacian + source))

    # Data loss
    T_pred = model(xy_data)
    data_loss = torch.mean(torch.square(T_pred - T_data))

    return physics_loss + 10.0 * data_loss

# Training loop
epochs = 5000
for epoch in range(epochs):
    optimizer.zero_grad()
    loss = total_loss(model, xy_mesh, xy_data, T_data)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 500 == 0:
        print(f'Epoch {epoch+1}')
        print(f'Loss: {loss.item():.6f}')
        print(f'Predicted source location: ({model.source_x.item():.3f}, {model.source_y.item():.3f})')
        print(f'Predicted strength: {model.source_strength.item():.3f}\n')

# Visualization
with torch.no_grad():
    T_pred = model(xy_mesh).numpy().reshape(50, 50)

    plt.figure(figsize=(15, 5))

    # Temperature field
    plt.subplot(1, 3, 1)
    plt.contourf(X, Y, T_pred, levels=20, cmap='hot')
    plt.colorbar(label='Temperature')
    plt.plot(x_data, y_data, 'ko', label='Sensors')
    plt.plot(model.source_x.item(), model.source_y.item(), 'r*',
             markersize=15, label='Predicted Source')
    plt.plot(0.7, 0.3, 'g*', markersize=15, label='True Source')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Temperature Field')
    plt.legend()

    # Source term
    plt.subplot(1, 3, 2)
    source = model.source_term(
        torch.tensor(X.flatten()),
        torch.tensor(Y.flatten())
    ).reshape(50, 50)
    plt.contourf(X, Y, source.numpy(), levels=20, cmap='viridis')
    plt.colorbar(label='Source Strength')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Heat Source Distribution')

    # Sensor measurements
    plt.subplot(1, 3, 3)
    plt.scatter(x_data, y_data, c=T_data.numpy(), cmap='hot')
    plt.colorbar(label='Temperature')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Sensor Measurements')

    plt.tight_layout()
    plt.show()

print("\nFinal Results:")
print(f"True source location: (0.700, 0.300)")
print(f"Predicted location: ({model.source_x.item():.3f}, {model.source_y.item():.3f})")
print(f"True strength: 2.000")
print(f"Predicted strength: {model.source_strength.item():.3f}")